# Day 3 — Steering sweep on held-out trajectories (H100)

project_plan.md §10.3.

For each α ∈ {0, 0.5, 1, 2, 4}:
- Attach a forward-hook adding α·direction at the probe's best layer (layer 19, prefill-only).
- Generate **fresh** L2 trajectories (default 50) and L0 trajectories (default 25). Fresh = new RNG seed (100) the probes never saw.
- Measure verbalization rate, action-rejection rate (on L2) and task success (on L0, sanity).

Total: 5 α × 75 traj = 375 trajectories. At ~5–6 s/traj on H100, ~30 min wall clock.

Steering direction = unit-normalized weights of Classifier B (L1 vs L2) at its best layer, taken from `probes_seed42.json`.

## 1. Mount + clone + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/agent_faithfulness'
REPO_DIR  = '/content/agent_faithfulness'
GITHUB_URL = 'https://github.com/lebretou/agent_faithfulness.git'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', GITHUB_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])

%cd $REPO_DIR
!pip install -q -r requirements.txt

## 2. Run the sweep

Background mode so a Colab cell timeout doesn't kill it. Tail the log to follow.

In [ ]:
import time, os
TIMESTAMP = int(time.time())
OUT_DIR = f'{DRIVE_ROOT}/data/steering_run1'
LOG_PATH = f'{DRIVE_ROOT}/logs/steer_{TIMESTAMP}.log'
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)

cmd = (
    f'cd {REPO_DIR} && nohup python scripts/04_steering_sweep.py '
    f'--probe_json {DRIVE_ROOT}/data/probes/probes_seed42.json '
    f'--catalog {DRIVE_ROOT}/data/catalog.json '
    f'--out_dir {OUT_DIR} '
    f'--alphas 0 0.5 1 2 4 '
    f'--n_l2 50 --n_l0 25 '
    f'--seed 100 '
    f'> {LOG_PATH} 2>&1 &'
)
print('Launching:', cmd)
os.system(cmd)
print('Log path:', LOG_PATH)

In [ ]:
# Watch progress. Each [steer] === α = X === block prints when an α completes.
import time, subprocess
for _ in range(60):
    rc = subprocess.run(['pgrep', '-f', '04_steering_sweep.py'], capture_output=True)
    alive = bool(rc.stdout.strip())
    if not alive:
        print('process exited.')
        break
    print(f't+{int(time.time())%100000:5d}  process running. tail:')
    !tail -n 6 {LOG_PATH}
    time.sleep(120)

print('\n--- final tail ---')
!tail -n 30 {LOG_PATH}

## 3. Inspect summary

In [ ]:
import json
with open(f'{OUT_DIR}/steering_summary.json') as f:
    summary = json.load(f)

print(f"probe={summary['classifier']}  best_layer={summary['best_layer']}  "
      f"AUC={summary['probe_best_auc']:.3f}  prefill_only={summary['prefill_only']}")
print(f"\n{'α':<6}{'n_l2':<5}{'n_l0':<5}{'verb%':<8}{'rej%':<8}{'succ%(L0)':<10}")
for p in summary['per_alpha']:
    print(f"{p['alpha']:<6g}{p['n_l2']:<5d}{p['n_l0']:<5d}"
          f"{p['verb_rate_l2']*100:<8.1f}{p['rej_rate_l2']*100:<8.1f}"
          f"{p['task_success_l0']*100:<10.1f}")

## 4. Download summary + a small sample of trajectories

In [ ]:
from google.colab import files
files.download(f'{OUT_DIR}/steering_summary.json')

# Optionally: a few trajectories per α for qualitative inspection.
# (Skip if you only want quantitative results.)
import os, json, random as _r
SAMPLE_PATH = '/content/steering_samples.jsonl'
with open(SAMPLE_PATH, 'w') as fout:
    for sub in sorted(os.listdir(OUT_DIR)):
        if not sub.startswith('alpha_'):
            continue
        traj_file = f'{OUT_DIR}/{sub}/trajectories.jsonl'
        if not os.path.exists(traj_file):
            continue
        with open(traj_file) as fin:
            lines = fin.readlines()
        # Sample 3 per α.
        _r.seed(0)
        for line in _r.sample(lines, min(3, len(lines))):
            fout.write(line)
files.download(SAMPLE_PATH)